# Session 1: why two cameras disagree
**A. Real-world problem.** You film a speckled rubber sheet with two cameras. Camera A looks
straight at it. Camera B stands to one side. B has a wider picture and runs at 24 pictures a
second while A runs at 30. When you put the two videos side by side, nothing lines up.

**Plan.** (1) draw a grid on a sheet and see it from two angles, (2) compare picture sizes
and shapes, (3) compare the two clocks, (4) match points by eye and find the stretch needed
to bring one picture onto the other.

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# Camera A: looks straight at the sheet. Camera B: stands 40 degrees to the side.
CAM_A = {"name": "A", "width": 640, "height": 480, "fps": 30, "yaw": 0.0, "px_per_unit": 40.0}
CAM_B = {"name": "B", "width": 1280, "height": 720, "fps": 24, "yaw": 40.0, "px_per_unit": 60.0}

## Step 1: one sheet, two views
The sheet is 10 units wide and 6 units tall, with a grid line every unit. A camera turned
`yaw` degrees to the side sees every sideways distance squeezed by cos(yaw); up-and-down
distances are not squeezed. (This is the simplest model: a far-away camera. Session 2 handles
the full perspective.) The picture also has a size: `px_per_unit` pixels for every unit.

In [ ]:
def grid_points(nx=11, ny=7):
    """Grid crossings on the sheet, in sheet units: (x, y) with x from 0 to 10 and y from 0 to 6."""
    gx, gy = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))
    return np.stack([gx.ravel(), gy.ravel()], axis=1)


def width_ratio(yaw_deg):
    """How much a sideways distance is squeezed when the camera is turned yaw_deg to the side."""
    return math.cos(math.radians(yaw_deg))


def view_of_sheet(points, cam):
    """Pixel positions (x right, y down) of sheet points, with the sheet centre in the picture centre."""
    pts = np.asarray(points, dtype=float) - np.array([5.0, 3.0])
    k = cam["px_per_unit"]
    x = cam["width"] / 2 + k * width_ratio(cam["yaw"]) * pts[:, 0]
    y = cam["height"] / 2 + k * pts[:, 1]
    return np.stack([x, y], axis=1)


def aspect_ratio(cam):
    return cam["width"] / cam["height"]

## Step 2: match points by eye, then find the stretch
Pick landmarks that you can recognise in both pictures (corners of the sheet). The sideways
stretch that brings picture B onto picture A is (distance between two landmarks in A) divided
by (the same distance in B). The up-and-down stretch is found the same way. If the two stretches
differ, a single "resize" cannot fix it.

In [ ]:
LANDMARKS = np.array([[0.0, 0.0], [10.0, 0.0], [10.0, 6.0], [0.0, 6.0]])      # the four sheet corners


def stretch_between(pa, pb):
    """(sideways, up-down) stretch that turns landmark spacing in picture B into spacing in picture A.
    pa and pb are arrays of the same landmarks; uses the first two for x and the first and last for y."""
    sx = (pa[1, 0] - pa[0, 0]) / (pb[1, 0] - pb[0, 0])
    sy = (pa[3, 1] - pa[0, 1]) / (pb[3, 1] - pb[0, 1])
    return float(sx), float(sy)

## Step 3: two clocks
Camera A takes a picture at 0, 1/30, 2/30 ... seconds; camera B at 0, 1/24, 2/24 ... seconds.
They only agree when k/30 = m/24, which happens every 1/6 s (6 is the largest number that
divides both 30 and 24). In between, the nearest B picture is up to some milliseconds away,
and a moving sample travels in that time.

In [ ]:
def frame_times(fps, seconds):
    return np.arange(int(math.floor(seconds * fps + 1e-9)) + 1) / fps


def coincide_period(fps_a, fps_b):
    """Seconds between moments when both cameras take a picture together (both rates whole numbers)."""
    return 1.0 / math.gcd(int(fps_a), int(fps_b))


def worst_gap(fps_a, fps_b, seconds=1.0):
    """Largest time distance (s) from any A picture to the nearest B picture."""
    ta = np.arange(int(seconds * fps_a)) / fps_a
    tb = np.arange(int(seconds * fps_b) + 2) / fps_b
    return float(max(np.min(np.abs(tb - t)) for t in ta))


def gap_to_b(t, fps_b):
    """Time distance (s) from instant t to the nearest camera B picture."""
    m = int(round(t * fps_b))
    return min(abs(t - c / fps_b) for c in range(max(0, m - 1), m + 2))


def position_error(speed, gap):
    """Distance (pixels) a sample moves during a timing gap, if you pretend the pictures are simultaneous."""
    return speed * gap

## See it
Left: the sheet in the two pictures (drawn to the same pixel scale, so you see which camera
has the bigger frame). Right: the picture times of both cameras on one time axis.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    pts = grid_points()
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
    for a, cam in zip(ax, (CAM_A, CAM_B)):
        p = view_of_sheet(pts, cam)
        a.add_patch(plt.Rectangle((0, 0), cam["width"], cam["height"], fc="none", ec="k"))
        a.plot(p[:, 0], p[:, 1], "b.", ms=4)
        lm = view_of_sheet(LANDMARKS, cam)
        for i, (x, y) in enumerate(lm, 1):
            a.plot(x, y, "ro")
            a.annotate(str(i), (x, y), textcoords="offset points", xytext=(5, 5), color="r")
        a.set_xlim(-20, 1300)
        a.set_ylim(740, -20)
        a.set_aspect("equal")
        a.set_title(f"camera {cam['name']}: {cam['width']} x {cam['height']} px, {aspect_ratio(cam):.2f}:1")
        a.set_xlabel("x (pixels)")
        a.set_ylabel("y (pixels)")
    fig.tight_layout()
    fig.savefig(OUT / "two_views.png", dpi=110)
    pa, pb = view_of_sheet(LANDMARKS, CAM_A), view_of_sheet(LANDMARKS, CAM_B)
    sx, sy = stretch_between(pa, pb)
    print(f"width ratio at yaw 40 degrees: {width_ratio(40):.3f}")
    print(f"stretch needed to bring B onto A: sideways {sx:.3f}, up-down {sy:.3f}")

    fig, ax = plt.subplots(figsize=(9, 2.8))
    ta, tb = frame_times(30, 0.2), frame_times(24, 0.2)
    ax.vlines(ta, 0.5, 1.0, color="tab:blue", label="camera A (30 fps)")
    ax.vlines(tb, 0.0, 0.5, color="tab:orange", label="camera B (24 fps)")
    ax.set_yticks([])
    ax.set_xlabel("time (s)")
    ax.set_title("Picture times: the cameras agree only at 0, 1/6 s ...")
    ax.legend(loc="upper right", ncol=2)
    fig.tight_layout()
    fig.savefig(OUT / "two_clocks.png", dpi=110)
    print(f"cameras take a picture together every {coincide_period(30, 24):.4f} s")
    gap = worst_gap(30, 24)
    print(f"worst gap between an A picture and the nearest B picture: {gap * 1000:.2f} ms")
    print(f"a sample moving at 120 px/s travels {position_error(120, gap):.1f} px in that time")
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** Change the yaw of camera B in `lesson.py` to 25 degrees and predict, before running, the new sideways stretch from B onto A.
2. **Stretch.** Write `agree_times(fps_a, fps_b, seconds)` that returns all the moments in one second when both cameras take a picture, and check it for 30 and 24 fps.
3. **Explain.** Explain why 30 fps and 60 fps never disagree but 30 fps and 25 fps do.

## Assessment
- **A. Conceptual.** Why can one resize factor not bring camera B's picture onto camera A's picture in this set-up?
- **B. Numerical.** A camera is turned 60 degrees to the side. A sideways distance is 100 pixels in a head-on picture. How long does it look? A sample at 90 px/s is paired with a picture 1/60 s away: how far has it moved?
- **C. Graph and simulation interpretation.** In the timing graph (frame rate of B = 24), what is the highest the curve reaches, and what do the red dots tell you?
- **D. Design and optimisation challenge.** Design: camera A runs at 30 fps. Which of these B frame rates keep the worst gap at or below 10 ms: 24, 25, 30, 48, 50, 60? Which would you choose, and what do you give up?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append((f"worked example: at 40 degrees a 100 px sideways distance looks {100 * L.width_ratio(40):.1f} px (cos 40 = 0.766)",
                   abs(100 * L.width_ratio(40) - 76.6) < 0.05))
    checks.append(("a camera looking straight on does not squeeze anything", abs(L.width_ratio(0) - 1.0) < 1e-12))
    checks.append(("edge: a camera at 90 degrees sees the sheet edge-on (squeeze 0)", abs(L.width_ratio(90)) < 1e-12))
    pa, pb = L.view_of_sheet(L.LANDMARKS, L.CAM_A), L.view_of_sheet(L.LANDMARKS, L.CAM_B)
    checks.append(("camera A shows the 10 x 6 sheet as 400 x 240 pixels",
                   abs(pa[1, 0] - pa[0, 0] - 400) < 1e-9 and abs(pa[3, 1] - pa[0, 1] - 240) < 1e-9))
    checks.append(("camera B shows it 459.6 pixels wide and 360 tall",
                   abs(pb[1, 0] - pb[0, 0] - 459.6) < 0.05 and abs(pb[3, 1] - pb[0, 1] - 360) < 1e-9))
    sx, sy = L.stretch_between(pa, pb)
    checks.append((f"the stretch from B onto A is {sx:.3f} sideways and {sy:.3f} up-down (hand calculation 40/(60 cos 40) and 40/60)",
                   abs(sx - 40 / (60 * math.cos(math.radians(40)))) < 1e-9 and abs(sy - 40 / 60) < 1e-9 and abs(sx - 0.870) < 0.001 and abs(sy - 0.667) < 0.001))
    checks.append(("one resize factor cannot fix it: the two stretches differ", abs(sx - sy) > 0.1))
    checks.append(("the grid has 11 x 7 = 77 crossings", len(L.grid_points()) == 77))
    checks.append(("the aspect ratios are 4:3 and 16:9", abs(L.aspect_ratio(L.CAM_A) - 4 / 3) < 1e-12 and abs(L.aspect_ratio(L.CAM_B) - 16 / 9) < 1e-12))
    checks.append(("30 fps over 0.2 s gives 7 picture times, 24 fps gives 5", len(L.frame_times(30, 0.2)) == 7 and len(L.frame_times(24, 0.2)) == 5))
    checks.append(("worked example: 30 fps and 24 fps agree every 1/6 s", abs(L.coincide_period(30, 24) - 1 / 6) < 1e-12))
    ta, tb = L.frame_times(30, 1.0), L.frame_times(24, 1.0)
    both = [t for t in ta if np.min(np.abs(tb - t)) < 1e-9]
    checks.append(("counting by hand over 1 s: exactly 6 shared instants (0, 1/6 ... 5/6) plus 1.0", len(both) == 7 and abs(both[1] - 1 / 6) < 1e-9))
    gap = L.worst_gap(30, 24)
    checks.append((f"worked example: the worst timing gap is 1/60 s = 16.7 ms (got {gap * 1000:.2f} ms)", abs(gap - 1 / 60) < 1e-9))
    checks.append(("a sample at 120 px/s moves 2.0 px in that gap", abs(L.position_error(120, gap) - 2.0) < 1e-6))
    checks.append(("edge: equal frame rates started together never disagree", L.worst_gap(30, 30) < 1e-12))
    checks.append(("a faster B camera shrinks the gap: 48 fps gives 8.33 ms", abs(L.worst_gap(30, 48) - 1 / 120) < 1e-9))
    checks.append(("a camera at 60 degrees squeezes sideways distances to half: 100 px look 50 px", abs(100 * L.width_ratio(60) - 50.0) < 1e-9))
    checks.append(("a sample at 90 px/s moves 1.5 px in the 1/60 s gap", abs(L.position_error(90, 1 / 60) - 1.5) < 1e-9))
    gaps = [round(L.gap_to_b(k / 30, 24) * 1000, 2) for k in range(5)]
    checks.append((f"the gaps at the five A pictures of one 1/6 s cycle are 0, 8.33, 16.67, 16.67, 8.33 ms (got {gaps})", gaps == [0.0, 8.33, 16.67, 16.67, 8.33]))
    ts = np.linspace(0, 0.2, 4001)
    peak = max(L.gap_to_b(t, 24) for t in ts)
    checks.append((f"the nearest-picture distance never exceeds half a B period: 1/48 s = 20.8 ms (got {peak * 1000:.2f})", abs(peak - 1 / 48) < 1e-4))
    ok_fps = [f for f in (24, 25, 30, 48, 50, 60) if L.worst_gap(30, f) <= 0.010 + 1e-12]
    checks.append((f"design: for a gap of at most 10 ms the B rates 30, 48, 50 and 60 work (got {ok_fps})", ok_fps == [30, 48, 50, 60]))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
